In [ ]:
from google.colab import drive
drive.mount('/content/drive')

! pip install jaxopt
! pip install corner
! pip install emcee

import os
# os.environ["JAX_ENABLE_X64"] = "True"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "true"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
print(os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"])

path = '/content/drive/MyDrive/SchwarMAX-analytic/'

import sys
sys.path.append(path)

from model_bar import *
from likelihoods_bar import *
from utils import *
from sample_from_density import sample_from_density_grid

import jax
import jax.numpy as jnp

import jax
# jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import jax.numpy.linalg as jnn
import jaxopt
import pandas as pd
import numpy as np
import scipy as sp
import pickle

from tqdm import tqdm

import emcee
import corner
import matplotlib.pyplot as plt

import time

from constants import EPSILON

print('jax version', jax.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
0.95
jax version 0.7.2


In [ ]:
path = '/content/drive/MyDrive/SchwarMAX-analytic/'
filename = 'mock_data/mock_Nbody_bar_XY_withRot_Nbins600_beta25_gamma110_D50_gal2.pkl'
# filename = 'mock_Nbody_bar_XY_withRot_gal2_Nbins1000.pkl'
dict_data = get_dict_data_bootstrap(path, filename, n_samples = 7_500)

def log_prior(theta,):
    if (7 < theta[0] < 12) and (8 < theta[1] < 12) and (-1 < theta[2] < 2) and (-1 < theta[3] < 1) and (-1 < theta[4] < 1)\
    and (0 <= theta[5] < jnp.pi) and (0 <= theta[6] < jnp.pi/2) and (0 <= theta[7] < jnp.pi):
        return 0.0  # log(1) = 0 for uniform prior
    return -np.inf  # log(0) = -inf for out-of-bounds

def log_prob(theta,):
    # print(theta)
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf

    ll = logl_density(theta, dict_data, dict_data['total_bins'])

    return ll + lp

ndim = 8
nwalkers = 16  # must be >= 2 * ndim

# Initialize walkers around ground truth
# p0 = np.array([ground_truth[k] for k in param_names])
p0 = np.array([10.5, 10, 0.8, 0., 0.5, jnp.pi/4, jnp.pi/4, 3.5*jnp.pi/4])
# initial_pos = p0 + 1e-1 * np.random.randn(nwalkers, ndim)
np.random.seed(42)
initial_pos = p0 + np.random.uniform(-0.3, 0.3, (nwalkers, ndim))

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob)
sampler.run_mcmc(initial_pos, 500, progress=True)

samples = sampler.get_chain(discard=200, flat=True)


params_bestfit = np.percentile(samples, axis=0, q=50)
logl_val = logl_density(params_bestfit, dict_data, dict_data['total_bins'])
print('Best-fit logL projection', logl_val)#

dict_data['logl_density_max'] = logl_val

logM_disc_best_fit, logM_bulge_best_fit, \
logRd_disc_best_fit, logHs_disc_best_fit, logRs_bulge_best_fit, \
alpha_best_fit, beta_best_fit, gamma_best_fit = params_bestfit
# logMhalo_best_fit, logrho0_best_fit, logM_bulge_best_fit, logRh_disk_best_fit, logRs_disk_best_fit, logHs_disk_best_fit, logRs_bulge_best_fit,\
#       alpha_best_fit, beta_best_fit, gamma_best_fit, logLM_best_fit = (11.8, 8.8, 10.4, 1.2, 0.45, -0.24, -0.1, 30*np.pi/180, 20*np.pi/180, 0*np.pi/180, 0)

print('logM_disc_best_fit',logM_disc_best_fit)
print('logM_bulge_best_fit',logM_bulge_best_fit)
print('logRd_disc_best_fit',logRd_disc_best_fit)
print('logHs_disc_best_fit',logHs_disc_best_fit)
print('logRs_bulge_best_fit',logRs_bulge_best_fit)
print('alpha_best_fit',alpha_best_fit * 180 / np.pi)
print('beta_best_fit',beta_best_fit * 180 / np.pi)
print('gamma_best_fit',gamma_best_fit * 180 / np.pi)


ground_truth = [
    10.779253,
    # 11.9,
    logM_disc_best_fit,
    logM_bulge_best_fit,
    0.8970275,
    # 1.276,
    logRd_disc_best_fit,
    logHs_disc_best_fit,
    logRs_bulge_best_fit,
    alpha_best_fit, #  10 * np.pi/180,
    beta_best_fit,
    gamma_best_fit,
    0.,
    1.5,

    0.6, # sigma_xy model term
]

vmap_direct = jax.vmap(logl_angular_input_bootstrap, in_axes=(0, None, None, None))
test_params = jnp.array([ground_truth]*4)
%time vmap_direct(test_params, dict_data, dict_data['total_bins'], dict_data['Rzphi_n_tot']).block_until_ready()
%time vmap_direct(test_params, dict_data, dict_data['total_bins'], dict_data['Rzphi_n_tot']).block_until_ready()

100%|██████████| 500/500 [00:09<00:00, 51.24it/s]


Best-fit logL projection -3.3157518
logM_disc_best_fit 10.694671130055296
logM_bulge_best_fit 9.926458886582076
logRd_disc_best_fit 0.6293224449307568
logHs_disc_best_fit 0.09200612385791021
logRs_bulge_best_fit -0.2611393465101124
alpha_best_fit 45.23673735949032
beta_best_fit 29.336115310463885
gamma_best_fit 119.28578479029026
CPU times: user 3min 20s, sys: 8.55 s, total: 3min 29s
Wall time: 2min 4s
CPU times: user 5.75 s, sys: 4.71 s, total: 10.5 s
Wall time: 10.5 s


Array([1027.502 , 1027.5182, 1027.554 , 1027.6132], dtype=float32)

In [ ]:
def parallel_minimize(logdensity_fn, x0s, spread, n_iter=200,
                      vmap_batch=20):
    """
    Parallel Nelder-Mead-inspired derivative-free optimizer.

    Each generation:
      1. Evaluate population in parallel (vmap)
      2. Replace worst points with reflections toward the global best
      3. All chains see the global best — communication!

    x0s: (n_particles, ndim) initial guesses
    """
    n_particles, ndim = x0s.shape

    _vmap_logp = jax.vmap(logdensity_fn)

    def eval_batched(positions):
        """Evaluate logP in batches of vmap_batch."""
        n = positions.shape[0]
        results = []
        for i in range(0, n, vmap_batch):
            batch = positions[i:i+vmap_batch]
            results.append(_vmap_logp(batch))
        return jnp.concatenate(results)

    # Initialise
    positions = jnp.array(x0s)
    logp = eval_batched(positions)

    best_idx = jnp.argmax(logp)
    global_best = positions[best_idx]
    global_best_logp = logp[best_idx]

    key = jax.random.PRNGKey(0)

    pbar = tqdm(range(n_iter), desc="Optimize")
    for gen in pbar:
        key, k1, k2 = jax.random.split(key, 3)

        # --- Generate proposals: move toward global best ---
        # Differential evolution style:
        #   proposal = current + F * (global_best - current) + noise
        F = 1.  # scale factor
        noise_scale = jax.random.normal(k1, positions.shape) * spread[None, :]

        proposals = positions + F * (global_best - positions) + noise_scale

        # # Also try some random exploration (20% of particles)
        # n_explore = max(n_particles // 5, 1)
        # explore_idx = jax.random.choice(k2, n_particles, (n_explore,),
        #                                   replace=False)
        # # Random perturbation around global best
        # explore_noise = jax.random.normal(
        #     k1, (n_explore, ndim)) * 0.3
        # proposals = proposals.at[explore_idx].set(
        #     global_best + explore_noise)

        # --- Evaluate proposals ---
        prop_logp = eval_batched(proposals)

        # --- Accept improvements ---
        improved = prop_logp > logp
        positions = jnp.where(improved[:, None], proposals, positions)
        logp = jnp.where(improved, prop_logp, logp)

        # --- Update global best ---
        gen_best_idx = jnp.argmax(logp)
        gen_best_logp = logp[gen_best_idx]
        if float(gen_best_logp) > float(global_best_logp):
            global_best = positions[gen_best_idx]
            global_best_logp = gen_best_logp

        pbar.set_postfix(
            best=f"{float(global_best_logp):.1f}",
            mean=f"{float(jnp.mean(logp)):.1f}",
            improved=f"{int(jnp.sum(improved))}/{n_particles}",
        )

    return np.array(global_best), float(global_best_logp)

# ground_truth = [
#     11.0,
#     logM_disc_best_fit,
#     logM_bulge_best_fit,
#     1.0,
#     logRd_disc_best_fit,
#     logHs_disc_best_fit,
#     # logRs_bulge_best_fit,
#     0.3,
#     alpha_best_fit, #  10 * np.pi/180,
#     beta_best_fit,
#     gamma_best_fit,
#     0.0,
#     1.6,

#     0.0, # sigma_xy model term
# ]

# BOUNDS_LO = jnp.array([
#     ground_truth[0] - 3, ground_truth[1] - 3, ground_truth[2] - 3,
#     ground_truth[3] - 1, ground_truth[4] - 1, ground_truth[5] - 1, max(0, ground_truth[6] - 1),
#     0., 0., 0., -2., 0., -1.,
# ])
# BOUNDS_HI = jnp.array([
#     ground_truth[0] + 3, ground_truth[1] + 3, ground_truth[2] + 3,
#     ground_truth[3] + 1, ground_truth[4] + 1, ground_truth[5] + 1, ground_truth[6] + 1,
#     float(jnp.pi), float(jnp.pi / 2), float(jnp.pi), 2., 2., 2.,
# ])

ground_truth = [
    11.0,
    logM_disc_best_fit,
    10.2,
    1.0,
    logRd_disc_best_fit,
    logHs_disc_best_fit,
    0.68,
    alpha_best_fit, #  10 * np.pi/180,
    beta_best_fit,
    gamma_best_fit,
    0.0,
    1.6,

    0.0, # sigma_xy model term
]


BOUNDS_LO = jnp.array([
    ground_truth[0] - 3, ground_truth[1] - 3, ground_truth[2] - 0.01,
    ground_truth[3] - 1, ground_truth[4] - 1, ground_truth[5] - 1, ground_truth[6] - 0.01,
    0., 0., 0., -2., 0., -1.,
])
BOUNDS_HI = jnp.array([
    ground_truth[0] + 3, ground_truth[1] + 3, ground_truth[2] + 0.01,
    ground_truth[3] + 1, ground_truth[4] + 1, ground_truth[5] + 1, ground_truth[6] + 0.01,
    float(jnp.pi), float(jnp.pi / 2), float(jnp.pi), 2., 2., 2.,
])

# ── Log-posterior ────────────────────────────────────────────────────
def logdensity_fn(theta):
    in_bounds = jnp.all((theta >= BOUNDS_LO) & (theta <= BOUNDS_HI))
    log_vol = jnp.sum(jnp.log(BOUNDS_HI - BOUNDS_LO))
    logprior = jnp.where(in_bounds, -log_vol, -jnp.inf)

    ll = logl_angular_input_bootstrap(theta, dict_data, dict_data['total_bins'], dict_data['Rzphi_n_tot'])
    ll = jnp.where(jnp.isfinite(ll), ll, -1e30)

    return logprior + ll

# 4 particles, vmap 4 at a time (one batch)
n_chain = 4
NDIM = len(ground_truth)

# init_spread = 0.05 * jnp.ones(NDIM)
init_spread = jnp.array([
    0.05, 0.05, 1e-4,
    0.05, 0.05, 0.05, 1e-4,
    0.05, 0.05, 0.05, 0.05, 0.05, 0.05
])

key = jax.random.PRNGKey(42)
x0s = jnp.array(ground_truth) + jax.random.normal(key, (n_chain, NDIM)) * init_spread[None, :]

best_params, best_logp = parallel_minimize(logdensity_fn, x0s, init_spread, n_iter=100)
print(np.round(best_params, 2), best_logp)

np.save(path+'/minimise_0418_Nbins600_beta25_gamma140_D50_gal2_fixedbar.npy', best_params)

Optimize: 100%|██████████| 100/100 [17:35<00:00, 10.56s/it, best=2914.8, improved=0/4, mean=2836.3]

[10.71 10.65 10.2   1.05  0.86 -0.04  0.68  0.8   0.4   2.36  0.07  1.43
  0.41] 2914.7568359375


In [ ]:

# ── Configuration ────────────────────────────────────────────────────
path = '/content/drive/MyDrive/SchwarMAX-analytic/'

N_WALKERS = 32             # must be even and >= 2*NDIM
N_STEPS = 500
CHECKPOINT_EVERY = 10
BURNIN = 300
STRETCH_A = 2.0            # stretch move scale parameter

MINIMISER_RESULT = os.path.join(path, 'minimise_0422_Nbins600_beta25_gamma110_D50_gal2.npy')
CHECKPOINT_FILE = os.path.join(path, 'ensemble_checkpoint_0423_beta25_gamma110_D50_gal2.pkl')
OUTPUT_FILE = os.path.join(path, 'ensemble_results_0423_beta25_gamma110_D50_gal2.pkl')
OUTPUT_CSV = os.path.join(path, 'ensemble_posterior_0423_beta25_gamma110_D50_gal2.csv')
# CHECKPOINT_FILE = os.path.join(path, 'ensemble_checkpoint_gal2_0413.pkl')
# OUTPUT_FILE = os.path.join(path, 'ensemble_results_gal2_0413.pkl')
# OUTPUT_CSV = os.path.join(path, 'ensemble_posterior_gal2_0413.csv')

# ── Best-fit point ───────────────────────────────────────────────────
# res = np.load(os.path.join(path, 'minimise_0329_gal2_Nbins1000.npy'))
# res[-1] = 0.5

# logM_10kpc, logC_halo = logM_logRs_to_logMenc_logc(res[0], res[3])
# print('logM_10kpc', 'logC_halo', logM_10kpc, logC_halo)

# res[0] = logM_10kpc
# res[3] = logC_halo


res = np.load(MINIMISER_RESULT)
res[-1] = 0.8333

res[-2] = 1.41
res[-4] = 110 * jnp.pi / 180
# ── Prior bounds (13D) ───────────────────────────────────────────────
NDIM = 13
param_names = [
    'logM_10kpc', 'logM_disk', 'logM_bar', 'logC_halo', 'logRs_disk',
    'logHs_disk', 'logL_bar', 'alpha', 'beta', 'gamma',
    'log_light_to_mass_ratio', 'log_Omega', 'log_sigma',
]

BOUNDS_LO = jnp.array([
    res[0] - 3, res[1] - 3, res[2] - 3, res[3] - 1,
    res[4] - 1, res[5] - 1, max(0, res[6] - 1),
    0., 0., 0., -2., 0., -1.,
])
BOUNDS_HI = jnp.array([
    res[0] + 3, res[1] + 3, res[2] + 3, res[3] + 1,
    res[4] + 1, res[5] + 1, res[6] + 1,
    float(jnp.pi), float(jnp.pi / 2), float(jnp.pi), 2., 2., 2.,
])

# BOUNDS_LO = jnp.array([
#     res[0] - 3, res[1] - 3, res[2] - 3, res[3] - 1,
#     res[4] - 1, res[5] - 1, max(0, res[6] - 1),
#     0., 0., 0., -2., 0., res[-1]-1e-3,
# ])
# BOUNDS_HI = jnp.array([
#     res[0] + 3, res[1] + 3, res[2] + 3, res[3] + 1,
#     res[4] + 1, res[5] + 1, res[6] + 1,
#     float(jnp.pi), float(jnp.pi / 2), float(jnp.pi), 2., 2., res[-1]+1e-3,
# ])

# ── Log-posterior ────────────────────────────────────────────────────
def logdensity_fn(theta):
    in_bounds = jnp.all((theta >= BOUNDS_LO) & (theta <= BOUNDS_HI))
    log_vol = jnp.sum(jnp.log(BOUNDS_HI - BOUNDS_LO))
    logprior = jnp.where(in_bounds, -log_vol, -jnp.inf)

    ll = logl_angular_input_bootstrap(theta, dict_data, dict_data['total_bins'], dict_data['Rzphi_n_tot'])
    ll = jnp.where(jnp.isfinite(ll), ll, -1e30)

    return logprior + ll

_vmap_logdensity = jax.vmap(logdensity_fn)

print('begining logL', logl_angular_input_bootstrap(res, dict_data, dict_data['total_bins'], dict_data['Rzphi_n_tot']))
# ── Move weights (emcee-style mixture) ──────────────────────────────
# Same mix as the user's emcee config:
#   70% DEMove, 20% DESnookerMove, 10% StretchMove
MOVE_WEIGHTS = jnp.array([0.7, 0.9, 1.0])  # cumulative thresholds

DE_GAMMA = 2.38 / jnp.sqrt(2.0 * NDIM)    # ter Braak (2006) optimal
DE_NOISE = 1e-5                              # small jitter for ergodicity
SNOOKER_GAMMA = 1.7                          # ter Braak & Vrugt (2008)


# ── Stretch move ────────────────────────────────────────────────────
def _sample_z(rng_key, n, a=STRETCH_A):
    """Sample Z from g(z) ∝ 1/sqrt(z) on [1/a, a]."""
    u = jax.random.uniform(rng_key, shape=(n,))
    return ((a - 1.0) * u + 1.0) ** 2 / a


def _stretch_propose(rng_key, active, complement, ndim):
    """Stretch move (Goodman & Weare 2010).
    Returns (proposals, log_factors)."""
    n_half = active.shape[0]
    k1, k2 = jax.random.split(rng_key)

    idx = jax.random.randint(k1, (n_half,), 0, complement.shape[0])
    companions = complement[idx]
    z = _sample_z(k2, n_half)

    proposals = companions + z[:, None] * (active - companions)
    log_factors = (ndim - 1) * jnp.log(z)
    return proposals, log_factors


# ── DE move (ter Braak 2006) ────────────────────────────────────────
def _de_propose(rng_key, active, complement, ndim,
                gamma=None, sigma=DE_NOISE):
    """Differential Evolution move.
    Y = X + gamma * (c[r1] - c[r2]) + noise
    Symmetric proposal → log_factor = 0."""
    n_half = active.shape[0]
    n_comp = complement.shape[0]
    k1, k2, k3 = jax.random.split(rng_key, 3)

    if gamma is None:
        gamma = 2.38 / jnp.sqrt(2.0 * ndim)

    # Pick two distinct companions per walker
    r1 = jax.random.randint(k1, (n_half,), 0, n_comp)
    r2 = jax.random.randint(k2, (n_half,), 0, n_comp - 1)
    # Avoid r1 == r2: shift r2 up by 1 where r2 >= r1
    r2 = jnp.where(r2 >= r1, r2 + 1, r2)

    diff = complement[r1] - complement[r2]
    noise = sigma * jax.random.normal(k3, active.shape)

    proposals = active + gamma * diff + noise
    log_factors = jnp.zeros(n_half)  # symmetric proposal
    return proposals, log_factors


# ── DE-Snooker move (ter Braak & Vrugt 2008) ───────────────────────
def _snooker_propose(rng_key, active, complement, ndim,
                     gamma=SNOOKER_GAMMA):
    """DE-Snooker move: project differential onto line through walker and pivot.
    Has Jacobian factor (||Y-z0|| / ||X-z0||)^(D-1)."""
    n_half = active.shape[0]
    n_comp = complement.shape[0]
    k1, k2, k3 = jax.random.split(rng_key, 3)

    # Pick 3 companions: z0 (pivot), z1, z2 (for differential)
    r0 = jax.random.randint(k1, (n_half,), 0, n_comp)
    r12 = jax.random.randint(k2, (n_half, 2), 0, n_comp)
    r1, r2 = r12[:, 0], r12[:, 1]

    z0 = complement[r0]   # pivot
    z1 = complement[r1]
    z2 = complement[r2]

    # Direction: active → pivot
    direction = active - z0                           # (n_half, ndim)
    dist = jnp.linalg.norm(direction, axis=1, keepdims=True)
    dist_safe = jnp.maximum(dist, 1e-30)
    d_hat = direction / dist_safe                     # unit vector

    # Differential projected onto direction
    diff = z1 - z2                                     # (n_half, ndim)
    proj = jnp.sum(diff * d_hat, axis=1, keepdims=True)  # scalar projection

    # Propose along the direction
    proposals = active + gamma * proj * d_hat

    # Jacobian: (||Y - z0|| / ||X - z0||)^(D-1)
    new_dist = jnp.linalg.norm(proposals - z0, axis=1)
    log_factors = (ndim - 1) * jnp.log(new_dist / dist_safe.squeeze())

    return proposals, log_factors


# ── Mixed move: per-walker random selection ─────────────────────────
def _mixed_move_half(rng_key, active, active_logp, complement):
    """Update active walkers using a mixture of DE, Snooker, and Stretch moves.

    Each walker independently draws which move to use, then all proposals
    are evaluated in a single vmapped logL batch.
    """
    n_half = active.shape[0]
    k_sel, k_de, k_snk, k_str, k_acc = jax.random.split(rng_key, 5)

    # Generate proposals from ALL three moves (compute all, select per-walker)
    prop_de, lf_de = _de_propose(k_de, active, complement, NDIM)
    prop_snk, lf_snk = _snooker_propose(k_snk, active, complement, NDIM)
    prop_str, lf_str = _stretch_propose(k_str, active, complement, NDIM)

    # Per-walker move selection
    u_sel = jax.random.uniform(k_sel, (n_half,))
    use_de = u_sel < MOVE_WEIGHTS[0]                                    # 0.0–0.7
    use_snk = (u_sel >= MOVE_WEIGHTS[0]) & (u_sel < MOVE_WEIGHTS[1])   # 0.7–0.9
    # else: stretch                                                      # 0.9–1.0

    # Select proposal and log_factor per walker
    proposals = jnp.where(use_de[:, None], prop_de,
                    jnp.where(use_snk[:, None], prop_snk, prop_str))
    log_factors = jnp.where(use_de, lf_de,
                    jnp.where(use_snk, lf_snk, lf_str))

    # Evaluate all proposals in one vmapped batch
    prop_logp = _vmap_logdensity(proposals)

    # Metropolis acceptance
    log_accept = log_factors + prop_logp - active_logp
    log_u = jnp.log(jax.random.uniform(k_acc, (n_half,)))
    accept = log_u < log_accept

    new_active = jnp.where(accept[:, None], proposals, active)
    new_logp = jnp.where(accept, prop_logp, active_logp)
    n_accepted = jnp.sum(accept)

    return new_active, new_logp, n_accepted


def ensemble_step(rng_key, positions, logp):
    """One full ensemble step: update both halves with mixed moves.

    Args:
        rng_key: JAX PRNG key
        positions: (N_WALKERS, NDIM)
        logp: (N_WALKERS,)

    Returns:
        new_positions, new_logp, n_accepted
    """
    n_half = N_WALKERS // 2
    k1, k2 = jax.random.split(rng_key)

    s0, s1 = positions[:n_half], positions[n_half:]
    lp0, lp1 = logp[:n_half], logp[n_half:]

    # Phase 1: update S0 using S1
    s0, lp0, acc0 = _mixed_move_half(k1, s0, lp0, s1)

    # Phase 2: update S1 using updated S0
    s1, lp1, acc1 = _mixed_move_half(k2, s1, lp1, s0)

    new_positions = jnp.concatenate([s0, s1], axis=0)
    new_logp = jnp.concatenate([lp0, lp1], axis=0)
    n_accepted = acc0 + acc1

    return new_positions, new_logp, n_accepted


# ── Initial positions ────────────────────────────────────────────────
def make_init_positions(rng_key):
    """Start walkers spread across a broad region around the best-fit.

    Uses ~30% of the prior width per parameter so walkers explore widely
    from the start — the starting point may not be the true mode.
    """
    p0 = jnp.array(res)
    # Spread = 10% of prior half-width per parameter
    init_spread = 0.1 * jnp.ones(len(BOUNDS_HI))# * (BOUNDS_HI - BOUNDS_LO) / 2.0
    noise = jax.random.normal(rng_key, shape=(N_WALKERS, NDIM)) * init_spread[None, :]
    positions = p0[None, :] + noise
    positions = jnp.clip(positions, BOUNDS_LO[None, :]+1e-5, BOUNDS_HI[None, :]-1e-5)
    return positions


# ── Checkpointing ───────────────────────────────────────────────────
def save_checkpoint(all_positions, all_logprob, step, rng_key):
    ckpt = {
        'all_samples': [np.array(s) for s in all_positions],
        'all_logprob': [np.array(lp) for lp in all_logprob],
        'step': step,
        'rng_key': np.array(rng_key),
    }
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(ckpt, f)


def load_checkpoint():
    if not os.path.exists(CHECKPOINT_FILE):
        return None
    with open(CHECKPOINT_FILE, 'rb') as f:
        ckpt = pickle.load(f)
    n_steps = len(ckpt['all_samples'])
    n_walkers = ckpt['all_samples'][0].shape[0]
    print(f"Found checkpoint: {n_steps} steps, {n_walkers} walkers")
    return ckpt


# ── Main loop ────────────────────────────────────────────────────────
def run_ensemble(resume=True):
    rng_key = jax.random.PRNGKey(42)

    ckpt = load_checkpoint() if resume else None

    if ckpt is not None:
        all_positions = ckpt['all_samples']
        all_logprob = ckpt['all_logprob']
        start_step = ckpt['step']
        rng_key = jnp.array(ckpt['rng_key'])

        positions = jnp.array(all_positions[-1])
        logp = jnp.array(all_logprob[-1])

        print(f"Resumed from step {start_step}, {len(all_positions)} samples")
    else:
        rng_key, init_key = jax.random.split(rng_key)
        positions = make_init_positions(init_key)

        print(f"Initialising {N_WALKERS} walkers...")
        n_half = N_WALKERS // 2
        logp = jnp.concatenate([
            _vmap_logdensity(positions[:n_half]),
            _vmap_logdensity(positions[n_half:]),
        ])
        print(f"  Init done. logP: mean={float(jnp.mean(logp)):.1f}, "
              f"max={float(jnp.max(logp)):.1f}")

        all_positions = [np.array(positions)]
        all_logprob = [np.array(logp)]
        start_step = 0

    print(f"\nEnsemble MCMC: {N_WALKERS} walkers, {N_STEPS} steps, {NDIM}D")
    print(f"  Stretch parameter a={STRETCH_A}")
    print(f"  Starting from step {start_step + 1}")

    pbar = tqdm(range(start_step + 1, N_STEPS + 1), desc="Ensemble", unit="step")
    for step in pbar:
        rng_key, step_key = jax.random.split(rng_key)

        positions, logp, n_accepted = ensemble_step(step_key, positions, logp)

        all_positions.append(np.array(positions))
        all_logprob.append(np.array(logp))

        if step % 5 == 0:
            acc = float(n_accepted) / N_WALKERS
            pbar.set_postfix(
                logP=f"{float(jnp.mean(logp)):.1f}",
                acc=f"{acc:.3f}",
                best=f"{float(jnp.max(logp)):.1f}",
            )

        if step % CHECKPOINT_EVERY == 0:
            save_checkpoint(all_positions, all_logprob, step, rng_key)

    # ── Results ──────────────────────────────────────────────────────
    chain = np.stack(all_positions, axis=0)  # (N_STEPS+1, N_WALKERS, NDIM)
    logprob = np.stack(all_logprob, axis=0)
    print(f"\nChain shape: {chain.shape}")

    if chain.shape[0] > BURNIN:
        flat_samples = chain[BURNIN:].reshape(-1, NDIM)
    else:
        flat_samples = chain.reshape(-1, NDIM)

    print(f"\n── Posterior summary ({flat_samples.shape[0]} samples) ──")
    print(f"{'Parameter':>30s} {'mean':>10s} {'std':>10s} "
          f"{'2.5%':>10s} {'97.5%':>10s} {'truth':>10s}")
    print("-" * 82)
    for i, name in enumerate(param_names):
        s = flat_samples[:, i]
        truth = res[i]
        print(f"{name:>30s} {s.mean():10.4f} {s.std():10.4f} "
              f"{np.percentile(s, 2.5):10.4f} {np.percentile(s, 97.5):10.4f} "
              f"{truth:10.4f}")

    results = {
        'chain': chain,
        'logprob': logprob,
        'flat_samples': flat_samples,
        'param_names': param_names,
    }
    with open(OUTPUT_FILE, 'wb') as f:
        pickle.dump(results, f)
    print(f"\nResults saved to {OUTPUT_FILE}")

    pd.DataFrame(flat_samples, columns=param_names).to_csv(OUTPUT_CSV, index=False)
    print(f"CSV saved to {OUTPUT_CSV}")

    return chain, logprob


if __name__ == '__main__':
    run_ensemble(resume=False)

begining logL 985.2314
Initialising 32 walkers...
  Init done. logP: mean=-31250000470233319371146526720.0, max=1428.6

Ensemble MCMC: 32 walkers, 500 steps, 13D
  Stretch parameter a=2.0
  Starting from step 1


Ensemble: 100%|██████████| 500/500 [8:44:57<00:00, 63.00s/step, acc=0.031, best=3616.2, logP=3592.7]


Chain shape: (501, 32, 13)

── Posterior summary (6432 samples) ──
                     Parameter       mean        std       2.5%      97.5%      truth
----------------------------------------------------------------------------------
                    logM_10kpc    10.8816     0.0213    10.8513    10.9456    10.6631
                     logM_disk    10.8821     0.0656    10.6787    11.0040    10.7649
                      logM_bar    10.1276     0.0193    10.0727    10.1668    10.0998
                     logC_halo     0.8731     0.1547     0.5372     1.1544     0.8826
                    logRs_disk     1.2098     0.0254     1.1509     1.2666     0.7693
                    logHs_disk    -0.1436     0.0199    -0.1982    -0.1018    -0.1328
                      logL_bar     0.5986     0.0178     0.5687     0.6458     0.4746
                         alpha     0.7102     0.0119     0.6866     0.7251     0.7381
                          beta     0.4208     0.0038     0.4122     0.4315 

# Fixed Bar property


In [ ]:

# ── Configuration ────────────────────────────────────────────────────
path = '/content/drive/MyDrive/SchwarMAX-analytic/'

N_WALKERS = 32             # must be even and >= 2*NDIM
N_STEPS = 1000
CHECKPOINT_EVERY = 10
BURNIN = 300
STRETCH_A = 2.0            # stretch move scale parameter

MINIMISER_RESULT = os.path.join(path, 'minimise_0418_Nbins600_beta20_gamma140_D50_gal2_fixedbar.npy')
CHECKPOINT_FILE = os.path.join(path, 'ensemble_checkpoint_0418_beta25_gamma140_D50_gal2_fixedbarlength.pkl')
OUTPUT_FILE = os.path.join(path, 'ensemble_results_0418_beta25_gamma140_D50_gal2_fixedbarlength.pkl')
OUTPUT_CSV = os.path.join(path, 'ensemble_posterior_0418_beta25_gamma140_D50_gal2_fixedbarlength.csv')
# CHECKPOINT_FILE = os.path.join(path, 'ensemble_checkpoint_gal2_0413.pkl')
# OUTPUT_FILE = os.path.join(path, 'ensemble_results_gal2_0413.pkl')
# OUTPUT_CSV = os.path.join(path, 'ensemble_posterior_gal2_0413.csv')

# ── Best-fit point ───────────────────────────────────────────────────
# res = np.load(os.path.join(path, 'minimise_0329_gal2_Nbins1000.npy'))
# res[-1] = 0.5

# logM_10kpc, logC_halo = logM_logRs_to_logMenc_logc(res[0], res[3])
# print('logM_10kpc', 'logC_halo', logM_10kpc, logC_halo)

# res[0] = logM_10kpc
# res[3] = logC_halo


res = np.load(MINIMISER_RESULT)
# ── Prior bounds (13D) ───────────────────────────────────────────────
NDIM = 13
param_names = [
    'logM_10kpc', 'logM_disk', 'logM_bar', 'logC_halo', 'logRs_disk',
    'logHs_disk', 'logL_bar', 'alpha', 'beta', 'gamma',
    'log_light_to_mass_ratio', 'log_Omega', 'log_sigma',
]

BOUNDS_LO = jnp.array([
    res[0] - 3, res[1] - 3, res[2] - 1,
    res[3] - 1, res[4] - 1, res[5] - 1, res[6] - 0.01,
    0., 0., 0., -2., 0., -1.,
])
BOUNDS_HI = jnp.array([
    res[0] + 3, res[1] + 3, res[2] + 1,
    res[3] + 1, res[4] + 1, res[5] + 1, res[6] + 0.01,
    float(jnp.pi), float(jnp.pi / 2), float(jnp.pi), 2., 2., 2.,
])

# ── Log-posterior ────────────────────────────────────────────────────
def logdensity_fn(theta):
    in_bounds = jnp.all((theta >= BOUNDS_LO) & (theta <= BOUNDS_HI))
    log_vol = jnp.sum(jnp.log(BOUNDS_HI - BOUNDS_LO))
    logprior = jnp.where(in_bounds, -log_vol, -jnp.inf)

    ll = logl_angular_input_bootstrap(theta, dict_data, dict_data['total_bins'], dict_data['Rzphi_n_tot'])
    ll = jnp.where(jnp.isfinite(ll), ll, -1e30)

    return logprior + ll

_vmap_logdensity = jax.vmap(logdensity_fn)

print('begining logL', logl_angular_input_bootstrap(res, dict_data, dict_data['total_bins'], dict_data['Rzphi_n_tot']))
# ── Move weights (emcee-style mixture) ──────────────────────────────
# Same mix as the user's emcee config:
#   70% DEMove, 20% DESnookerMove, 10% StretchMove
MOVE_WEIGHTS = jnp.array([0.7, 0.9, 1.0])  # cumulative thresholds

DE_GAMMA = 2.38 / jnp.sqrt(2.0 * NDIM)    # ter Braak (2006) optimal
DE_NOISE = 1e-5                              # small jitter for ergodicity
SNOOKER_GAMMA = 1.7                          # ter Braak & Vrugt (2008)


# ── Stretch move ────────────────────────────────────────────────────
def _sample_z(rng_key, n, a=STRETCH_A):
    """Sample Z from g(z) ∝ 1/sqrt(z) on [1/a, a]."""
    u = jax.random.uniform(rng_key, shape=(n,))
    return ((a - 1.0) * u + 1.0) ** 2 / a


def _stretch_propose(rng_key, active, complement, ndim):
    """Stretch move (Goodman & Weare 2010).
    Returns (proposals, log_factors)."""
    n_half = active.shape[0]
    k1, k2 = jax.random.split(rng_key)

    idx = jax.random.randint(k1, (n_half,), 0, complement.shape[0])
    companions = complement[idx]
    z = _sample_z(k2, n_half)

    proposals = companions + z[:, None] * (active - companions)
    log_factors = (ndim - 1) * jnp.log(z)
    return proposals, log_factors


# ── DE move (ter Braak 2006) ────────────────────────────────────────
def _de_propose(rng_key, active, complement, ndim,
                gamma=None, sigma=DE_NOISE):
    """Differential Evolution move.
    Y = X + gamma * (c[r1] - c[r2]) + noise
    Symmetric proposal → log_factor = 0."""
    n_half = active.shape[0]
    n_comp = complement.shape[0]
    k1, k2, k3 = jax.random.split(rng_key, 3)

    if gamma is None:
        gamma = 2.38 / jnp.sqrt(2.0 * ndim)

    # Pick two distinct companions per walker
    r1 = jax.random.randint(k1, (n_half,), 0, n_comp)
    r2 = jax.random.randint(k2, (n_half,), 0, n_comp - 1)
    # Avoid r1 == r2: shift r2 up by 1 where r2 >= r1
    r2 = jnp.where(r2 >= r1, r2 + 1, r2)

    diff = complement[r1] - complement[r2]
    noise = sigma * jax.random.normal(k3, active.shape)

    proposals = active + gamma * diff + noise
    log_factors = jnp.zeros(n_half)  # symmetric proposal
    return proposals, log_factors


# ── DE-Snooker move (ter Braak & Vrugt 2008) ───────────────────────
def _snooker_propose(rng_key, active, complement, ndim,
                     gamma=SNOOKER_GAMMA):
    """DE-Snooker move: project differential onto line through walker and pivot.
    Has Jacobian factor (||Y-z0|| / ||X-z0||)^(D-1)."""
    n_half = active.shape[0]
    n_comp = complement.shape[0]
    k1, k2, k3 = jax.random.split(rng_key, 3)

    # Pick 3 companions: z0 (pivot), z1, z2 (for differential)
    r0 = jax.random.randint(k1, (n_half,), 0, n_comp)
    r12 = jax.random.randint(k2, (n_half, 2), 0, n_comp)
    r1, r2 = r12[:, 0], r12[:, 1]

    z0 = complement[r0]   # pivot
    z1 = complement[r1]
    z2 = complement[r2]

    # Direction: active → pivot
    direction = active - z0                           # (n_half, ndim)
    dist = jnp.linalg.norm(direction, axis=1, keepdims=True)
    dist_safe = jnp.maximum(dist, 1e-30)
    d_hat = direction / dist_safe                     # unit vector

    # Differential projected onto direction
    diff = z1 - z2                                     # (n_half, ndim)
    proj = jnp.sum(diff * d_hat, axis=1, keepdims=True)  # scalar projection

    # Propose along the direction
    proposals = active + gamma * proj * d_hat

    # Jacobian: (||Y - z0|| / ||X - z0||)^(D-1)
    new_dist = jnp.linalg.norm(proposals - z0, axis=1)
    log_factors = (ndim - 1) * jnp.log(new_dist / dist_safe.squeeze())

    return proposals, log_factors


# ── Mixed move: per-walker random selection ─────────────────────────
def _mixed_move_half(rng_key, active, active_logp, complement):
    """Update active walkers using a mixture of DE, Snooker, and Stretch moves.

    Each walker independently draws which move to use, then all proposals
    are evaluated in a single vmapped logL batch.
    """
    n_half = active.shape[0]
    k_sel, k_de, k_snk, k_str, k_acc = jax.random.split(rng_key, 5)

    # Generate proposals from ALL three moves (compute all, select per-walker)
    prop_de, lf_de = _de_propose(k_de, active, complement, NDIM)
    prop_snk, lf_snk = _snooker_propose(k_snk, active, complement, NDIM)
    prop_str, lf_str = _stretch_propose(k_str, active, complement, NDIM)

    # Per-walker move selection
    u_sel = jax.random.uniform(k_sel, (n_half,))
    use_de = u_sel < MOVE_WEIGHTS[0]                                    # 0.0–0.7
    use_snk = (u_sel >= MOVE_WEIGHTS[0]) & (u_sel < MOVE_WEIGHTS[1])   # 0.7–0.9
    # else: stretch                                                      # 0.9–1.0

    # Select proposal and log_factor per walker
    proposals = jnp.where(use_de[:, None], prop_de,
                    jnp.where(use_snk[:, None], prop_snk, prop_str))
    log_factors = jnp.where(use_de, lf_de,
                    jnp.where(use_snk, lf_snk, lf_str))

    # Evaluate all proposals in one vmapped batch
    prop_logp = _vmap_logdensity(proposals)

    # Metropolis acceptance
    log_accept = log_factors + prop_logp - active_logp
    log_u = jnp.log(jax.random.uniform(k_acc, (n_half,)))
    accept = log_u < log_accept

    new_active = jnp.where(accept[:, None], proposals, active)
    new_logp = jnp.where(accept, prop_logp, active_logp)
    n_accepted = jnp.sum(accept)

    return new_active, new_logp, n_accepted


def ensemble_step(rng_key, positions, logp):
    """One full ensemble step: update both halves with mixed moves.

    Args:
        rng_key: JAX PRNG key
        positions: (N_WALKERS, NDIM)
        logp: (N_WALKERS,)

    Returns:
        new_positions, new_logp, n_accepted
    """
    n_half = N_WALKERS // 2
    k1, k2 = jax.random.split(rng_key)

    s0, s1 = positions[:n_half], positions[n_half:]
    lp0, lp1 = logp[:n_half], logp[n_half:]

    # Phase 1: update S0 using S1
    s0, lp0, acc0 = _mixed_move_half(k1, s0, lp0, s1)

    # Phase 2: update S1 using updated S0
    s1, lp1, acc1 = _mixed_move_half(k2, s1, lp1, s0)

    new_positions = jnp.concatenate([s0, s1], axis=0)
    new_logp = jnp.concatenate([lp0, lp1], axis=0)
    n_accepted = acc0 + acc1

    return new_positions, new_logp, n_accepted


# ── Initial positions ────────────────────────────────────────────────
def make_init_positions(rng_key):
    """Start walkers spread across a broad region around the best-fit.

    Uses ~30% of the prior width per parameter so walkers explore widely
    from the start — the starting point may not be the true mode.
    """
    p0 = jnp.array(res)
    # Spread = 10% of prior half-width per parameter
    init_spread = 0.1 * jnp.ones(len(BOUNDS_HI))# * (BOUNDS_HI - BOUNDS_LO) / 2.0
    noise = jax.random.normal(rng_key, shape=(N_WALKERS, NDIM)) * init_spread[None, :]
    positions = p0[None, :] + noise
    positions = jnp.clip(positions, BOUNDS_LO[None, :]+1e-3, BOUNDS_HI[None, :])-1e-3
    return positions


# ── Checkpointing ───────────────────────────────────────────────────
def save_checkpoint(all_positions, all_logprob, step, rng_key):
    ckpt = {
        'all_samples': [np.array(s) for s in all_positions],
        'all_logprob': [np.array(lp) for lp in all_logprob],
        'step': step,
        'rng_key': np.array(rng_key),
    }
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(ckpt, f)


def load_checkpoint():
    if not os.path.exists(CHECKPOINT_FILE):
        return None
    with open(CHECKPOINT_FILE, 'rb') as f:
        ckpt = pickle.load(f)
    n_steps = len(ckpt['all_samples'])
    n_walkers = ckpt['all_samples'][0].shape[0]
    print(f"Found checkpoint: {n_steps} steps, {n_walkers} walkers")
    return ckpt


# ── Main loop ────────────────────────────────────────────────────────
def run_ensemble(resume=True):
    rng_key = jax.random.PRNGKey(42)

    ckpt = load_checkpoint() if resume else None

    if ckpt is not None:
        all_positions = ckpt['all_samples']
        all_logprob = ckpt['all_logprob']
        start_step = ckpt['step']
        rng_key = jnp.array(ckpt['rng_key'])

        positions = jnp.array(all_positions[-1])
        logp = jnp.array(all_logprob[-1])

        print(f"Resumed from step {start_step}, {len(all_positions)} samples")
    else:
        rng_key, init_key = jax.random.split(rng_key)
        positions = make_init_positions(init_key)

        print(f"Initialising {N_WALKERS} walkers...")
        n_half = N_WALKERS // 2
        logp = jnp.concatenate([
            _vmap_logdensity(positions[:n_half]),
            _vmap_logdensity(positions[n_half:]),
        ])
        print(f"  Init done. logP: mean={float(jnp.mean(logp)):.1f}, "
              f"max={float(jnp.max(logp)):.1f}")

        all_positions = [np.array(positions)]
        all_logprob = [np.array(logp)]
        start_step = 0

    print(f"\nEnsemble MCMC: {N_WALKERS} walkers, {N_STEPS} steps, {NDIM}D")
    print(f"  Stretch parameter a={STRETCH_A}")
    print(f"  Starting from step {start_step + 1}")

    pbar = tqdm(range(start_step + 1, N_STEPS + 1), desc="Ensemble", unit="step")
    for step in pbar:
        rng_key, step_key = jax.random.split(rng_key)

        positions, logp, n_accepted = ensemble_step(step_key, positions, logp)

        all_positions.append(np.array(positions))
        all_logprob.append(np.array(logp))

        if step % 5 == 0:
            acc = float(n_accepted) / N_WALKERS
            pbar.set_postfix(
                logP=f"{float(jnp.mean(logp)):.1f}",
                acc=f"{acc:.3f}",
                best=f"{float(jnp.max(logp)):.1f}",
            )

        if step % CHECKPOINT_EVERY == 0:
            save_checkpoint(all_positions, all_logprob, step, rng_key)

    # ── Results ──────────────────────────────────────────────────────
    chain = np.stack(all_positions, axis=0)  # (N_STEPS+1, N_WALKERS, NDIM)
    logprob = np.stack(all_logprob, axis=0)
    print(f"\nChain shape: {chain.shape}")

    if chain.shape[0] > BURNIN:
        flat_samples = chain[BURNIN:].reshape(-1, NDIM)
    else:
        flat_samples = chain.reshape(-1, NDIM)

    print(f"\n── Posterior summary ({flat_samples.shape[0]} samples) ──")
    print(f"{'Parameter':>30s} {'mean':>10s} {'std':>10s} "
          f"{'2.5%':>10s} {'97.5%':>10s} {'truth':>10s}")
    print("-" * 82)
    for i, name in enumerate(param_names):
        s = flat_samples[:, i]
        truth = res[i]
        print(f"{name:>30s} {s.mean():10.4f} {s.std():10.4f} "
              f"{np.percentile(s, 2.5):10.4f} {np.percentile(s, 97.5):10.4f} "
              f"{truth:10.4f}")

    results = {
        'chain': chain,
        'logprob': logprob,
        'flat_samples': flat_samples,
        'param_names': param_names,
    }
    with open(OUTPUT_FILE, 'wb') as f:
        pickle.dump(results, f)
    print(f"\nResults saved to {OUTPUT_FILE}")

    pd.DataFrame(flat_samples, columns=param_names).to_csv(OUTPUT_CSV, index=False)
    print(f"CSV saved to {OUTPUT_CSV}")

    return chain, logprob


if __name__ == '__main__':
    run_ensemble(resume=False)

begining logL 2896.3206
Initialising 32 walkers...
  Init done. logP: mean=-31250000470233319371146526720.0, max=2803.8

Ensemble MCMC: 32 walkers, 1000 steps, 13D
  Stretch parameter a=2.0
  Starting from step 1


Ensemble: 100%|██████████| 1000/1000 [17:34:46<00:00, 63.29s/step, acc=0.000, best=3310.9, logP=3290.5]



Chain shape: (1001, 32, 13)

── Posterior summary (22432 samples) ──
                     Parameter       mean        std       2.5%      97.5%      truth
----------------------------------------------------------------------------------
                    logM_10kpc    10.7521     0.0113    10.7262    10.7728    10.7053
                     logM_disk    10.6815     0.0144    10.6538    10.7052    10.6548
                      logM_bar    10.1322     0.0111    10.1092    10.1503    10.2008
                     logC_halo     1.1110     0.0554     0.9962     1.2458     1.0541
                    logRs_disk     0.9729     0.0176     0.9427     1.0068     0.8605
                    logHs_disk    -0.0510     0.0128    -0.0717    -0.0257    -0.0353
                      logL_bar     0.6716     0.0009     0.6706     0.6740     0.6804
                         alpha     0.7106     0.0137     0.6813     0.7392     0.8043
                          beta     0.4402     0.0052     0.4280     0.448

# Jackknife error scale

In [ ]:
def load_checkpoint(filepath):
    with open(filepath, 'rb') as f:
        ckpt = pickle.load(f)

    all_samples = ckpt['all_samples']   # list of (N_CHAINS, NDIM) arrays
    all_logprob = ckpt['all_logprob']   # list of (N_CHAINS,) arrays
    step = ckpt['step']


    # Stack into (N_STEPS, N_CHAINS, NDIM) and (N_STEPS, N_CHAINS)
    posterior = np.stack(all_samples, axis=0)
    logprob = np.stack(all_logprob, axis=0)

    print(f"Loaded checkpoint: {step} steps, "
          f"{posterior.shape[1]} chains, {posterior.shape[2]} params")
    print(f"Chain shape: {posterior.shape}")
    return posterior, logprob, step



path = '/content/drive/MyDrive/SchwarMAX-analytic/'
filename = 'mock_data/mock_Nbody_bar_XY_withRot_Nbins600_beta25_gamma140_D50_gal2.pkl'
# filename = 'mock_Nbody_bar_XY_withRot_gal2_Nbins1000.pkl'
dict_data = get_dict_data_bootstrap(path, filename, n_samples = 7_500)

MINIMISER_RESULT = os.path.join(path, 'minimise_0415_Nbins600_beta25_gamma140_D50_gal2.npy')
res = np.load(MINIMISER_RESULT)

res = jnp.array(res)

# CHECKPOINT_FILE = path+'/ensemble_checkpoint_0415_beta25_gamma140_D50_gal2.pkl'


# posterior, logprob, step = load_checkpoint(CHECKPOINT_FILE)
# posterior = posterior[:, logprob[-1, :]>np.amax(logprob[-1, :])-100, :]
# posterior = posterior[300::10, :, :]
# posterior = posterior.reshape(-1, posterior.shape[-1])


# best_fit_param = np.percentile(posterior, 50, axis=0)


# logMhalo_10_best_fit, logMdisk_best_fit, logMbar_best_fit, logC_halo_best_fit, logRs_disk_best_fit, logHs_disk_best_fit, logRs_bar_best_fit,\
#       alpha_best_fit, beta_best_fit, gamma_best_fit, logLM_best_fit, logOmega_best_fit, logSigma_amplifier_best_fit = best_fit_param

# res = best_fit_param
print(np.round(res, 2))

chi2_full, chi2_all, chi2_mean, _, chi2_jack_std, _, _ = jackknife_error_wrapper(res, dict_data, dict_data['total_bins'], dict_data['Rzphi_n_tot'], n_groups = 600)

print(chi2_full, chi2_mean, chi2_jack_std)
print(np.std(chi2_all))

[10.67       10.75       10.05        1.18        0.84999996 -0.17
  0.48999998  0.71        0.45999998  2.6         0.08        1.4399999
  0.29999998]
18216.87 18180.514 1333.1696
51.776012


In [ ]:
np.std(chi2_all)

Array(51.75906, dtype=float32)